In [ ]:
import nltk
import re
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# If you get "punkt not found", run this once:
# nltk.download("punkt")

# ---------------------------------------------------------
# 1) Read text file - one sentence per line
# ---------------------------------------------------------
def read_file_lines(file_path):
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line != "":
                data.append(line)
    return data


# ---------------------------------------------------------
# 2) Read your gold file format (supports 1 or 2 occupations)
# ---------------------------------------------------------
def parse_gold_line(line):
    parts = line.strip().split("\t")
    if len(parts) < 3:
        raise ValueError("Gold line does not have 3 TAB-separated columns:\n" + line)

    label_part = parts[0].strip()
    ref_sentence = parts[1].strip()
    occ_part = parts[2].strip()

    # occupations
    occs = occ_part.split("|")
    occA = occs[0].strip().lower()
    occB = occs[1].strip().lower() if len(occs) > 1 else None

    # labels (gender@pos)
    labels = label_part.split("|")

    # A
    genderA, posA = labels[0].split("@")
    genderA = genderA.strip().lower()
    posA = int(posA.strip())

    # B (optional)
    genderB, posB = None, None
    if len(labels) > 1:
        genderB, posB = labels[1].split("@")
        genderB = genderB.strip().lower()
        posB = int(posB.strip())

    return {
        "goldA": genderA,
        "posA": posA,
        "occA": occA,
        "goldB": genderB,
        "posB": posB,
        "occB": occB,
        "ref": ref_sentence
    }


def read_gold_twoocc(file_path):
    rows = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line == "":
                continue
            rows.append(parse_gold_line(line))
    return rows


# ---------------------------------------------------------
# 3) Prediction logic (translation-driven),
# but filtering/conditioning is gold-only.
# ---------------------------------------------------------
male_words = {"he", "him", "his"}
female_words = {"she", "her", "hers"}


def find_or_prompt_occ(sentence, occ, gold_sentence=None, occ_map_cache=None, occ_skip_cache=None):
    s = sentence.lower()
    occ = occ.lower().strip()

    candidates = [occ]
    if occ_map_cache is not None:
        occ_map_cache.setdefault(occ, set())
        candidates.extend(sorted(list(occ_map_cache[occ])))

    seen = set()
    candidates_unique = []
    for c in candidates:
        c = c.strip().lower()
        if c and c not in seen:
            seen.add(c)
            candidates_unique.append(c)

    for cand in candidates_unique:
        pattern = r"\b" + re.escape(cand) + r"\b"
        matches = list(re.finditer(pattern, s))
        if matches:
            return cand, matches

    if occ_skip_cache is not None and occ in occ_skip_cache:
        return None, None

    print("\n" + "=" * 70)
    print("OCCUPATION NOT FOUND IN TRANSLATION")
    print("=" * 70)
    if gold_sentence is not None:
        print("Gold sentence   :", gold_sentence)
    print("Gold occupation :", occ)
    print("Translated sent :", sentence)
    print("\nType the occupation word/phrase used in the TRANSLATION for this role.")
    print("If you want to SKIP future 'not found' cases for this gold occupation, press Enter (blank) or type esc.")

    user_occ_raw = input("Enter translated occupation: ")
    user_occ = (user_occ_raw or "").strip().lower()

    if user_occ == "" or user_occ == "esc" or user_occ_raw == "\x1b":
        if occ_skip_cache is not None:
            occ_skip_cache.add(occ)
        return None, None

    if occ_map_cache is not None:
        occ_map_cache.setdefault(occ, set()).add(user_occ)

    pattern = r"\b" + re.escape(user_occ) + r"\b"
    matches = list(re.finditer(pattern, s))

    if not matches:
        return None, None

    return user_occ, matches


def predict_label_using_occ_robust(
    sentence,
    occ,
    window_size=12,
    gold_sentence=None,
    is_single_occ=False,
    occ_map_cache=None,
    occ_skip_cache=None
):
    s = sentence.lower()
    occ = occ.lower().strip()

    if is_single_occ:
        chosen_occ, chosen_matches = find_or_prompt_occ(
            sentence, occ, gold_sentence=gold_sentence,
            occ_map_cache=occ_map_cache, occ_skip_cache=occ_skip_cache
        )
        if chosen_occ is None:
            return None, None

        for match in chosen_matches:
            after_text = s[match.end():]
            tokens = nltk.word_tokenize(after_text)
            for tok in tokens[:window_size]:
                if tok in male_words:
                    return "male", chosen_occ
                if tok in female_words:
                    return "female", chosen_occ

        tokens_all = nltk.word_tokenize(s)
        has_male = any(t in male_words for t in tokens_all)
        has_female = any(t in female_words for t in tokens_all)

        if has_male and not has_female:
            return "male", chosen_occ
        if has_female and not has_male:
            return "female", chosen_occ

        return "neutral", chosen_occ

    chosen_occ, chosen_matches = find_or_prompt_occ(
        sentence, occ, gold_sentence=gold_sentence,
        occ_map_cache=occ_map_cache, occ_skip_cache=occ_skip_cache
    )
    if chosen_occ is None:
        return None, None

    for match in chosen_matches:
        after_text = s[match.end():]
        tokens = nltk.word_tokenize(after_text)
        for tok in tokens[:window_size]:
            if tok in male_words:
                return "male", chosen_occ
            if tok in female_words:
                return "female", chosen_occ

    tokens_all = nltk.word_tokenize(s)
    has_male = any(t in male_words for t in tokens_all)
    has_female = any(t in female_words for t in tokens_all)

    if has_male and not has_female:
        return "male", chosen_occ
    if has_female and not has_male:
        return "female", chosen_occ

    return "neutral", chosen_occ


# ---------------------------------------------------------
# 3.5) GOLD-ONLY FILTER: explicit pronoun near BOTH gold occupations
# ---------------------------------------------------------
def _find_phrase_starts(tokens, phrase_tokens):
    starts = []
    n = len(tokens)
    m = len(phrase_tokens)
    if m == 0 or n < m:
        return starts
    for i in range(n - m + 1):
        if tokens[i:i+m] == phrase_tokens:
            starts.append(i)
    return starts


def _has_gender_pronoun_near(tokens, occ_phrase, gold_gender, window=6):
    if occ_phrase is None or gold_gender is None:
        return False

    occ_tokens = nltk.word_tokenize(occ_phrase.lower())
    starts = _find_phrase_starts(tokens, occ_tokens)
    if not starts:
        return False

    for s_idx in starts:
        left = max(0, s_idx - window)
        right = min(len(tokens), s_idx + len(occ_tokens) + window)
        span = tokens[left:right]

        if gold_gender == "male":
            if any(t in male_words for t in span):
                return True
        elif gold_gender == "female":
            if any(t in female_words for t in span):
                return True

    return False


def gold_has_explicit_pronoun_near_both_occs(row, window=6):
    if row["occB"] is None or row["goldB"] is None:
        return False

    ref = row["ref"].lower()
    tokens = nltk.word_tokenize(ref)

    okA = _has_gender_pronoun_near(tokens, row["occA"], row["goldA"], window=window)
    okB = _has_gender_pronoun_near(tokens, row["occB"], row["goldB"], window=window)
    return okA and okB


# ---------------------------------------------------------
# Pretty-print examples used for a subset
# ---------------------------------------------------------
def print_subset_examples(title, examples, max_print=10):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)
    if not examples:
        print("None ✅")
        return

    to_show = examples[:max_print]
    for ex in to_show:
        print(f"\n--- Example line {ex['index']} ---")
        print("Gold A / B :", ex["goldA"], "/", ex["goldB"])
        print("Pred A / B :", ex["predA"], "/", ex["predB"])
        print("Occ  A / B :", ex["occA"], "/", ex["occB"])
        # optional debug: show what was actually matched (gold or mapped)
        print("Chosen occA/B:", ex.get("chosen_occ_A"), "/", ex.get("chosen_occ_B"))
        print("Gold ref   :", ex["ref"])
        print("Translation:", ex["translation"])

    if len(examples) > max_print:
        print(f"\n(Showing first {max_print}. Total examples in this subset: {len(examples)})")


# ---------------------------------------------------------
# 4) Evaluate rows + conditional subsets + store example rows
#    ✅ Role-wise + combined scoring unchanged
#    ✅ Interaction subsets ONLY: discard ENTIRE sentence if pA is None or pB is None
# ---------------------------------------------------------
def evaluate_twoocc(system_outputs, gold_rows, window_size=12, max_print=20, gold_pron_window=6, subset_example_print=10):
    if len(system_outputs) != len(gold_rows):
        raise ValueError(
            f"Line count mismatch!\n"
            f"System outputs: {len(system_outputs)}\n"
            f"Gold rows     : {len(gold_rows)}\n"
            f"They must be aligned line-by-line."
        )

    goldA, predA = [], []
    goldB, predB = [], []

    # conditional B|A
    B_gold_given_A_male, B_pred_given_A_male = [], []
    B_gold_given_A_female, B_pred_given_A_female = [], []

    # conditional A|B
    A_gold_given_B_male, A_pred_given_B_male = [], []
    A_gold_given_B_female, A_pred_given_B_female = [], []

    examples_B_given_A_male = []
    examples_B_given_A_female = []
    examples_A_given_B_male = []
    examples_A_given_B_female = []

    occ_map_cache = {}
    occ_skip_cache = set()

    neutral_total = 0
    neutral_actually_male = 0
    neutral_actually_female = 0

    neutral_male_examples = []
    neutral_female_examples = []
    skipped_examples = []

    gold_pron_pass = 0

    # ✅ NEW: interaction-only discard counter
    interaction_dropped_missing_role = 0

    for i, sent in enumerate(system_outputs):
        row = gold_rows[i]
        is_single = (row["occB"] is None or row["goldB"] is None)

        # ---- A ----
        gA = row["goldA"]
        oA = row["occA"]

        # ✅ CHANGE: capture chosen_occ_A (gold or mapped)
        pA, chosen_occ_A = predict_label_using_occ_robust(
            sent, oA,
            window_size=window_size,
            gold_sentence=row["ref"],
            is_single_occ=is_single,
            occ_map_cache=occ_map_cache,
            occ_skip_cache=occ_skip_cache
        )

        # role-wise scoring unchanged
        goldA.append(gA)
        predA.append(pA)

        if pA is None:
            skipped_examples.append({
                "role": "A",
                "index": i + 1,
                "gold": gA,
                "occ": oA,
                "gold_sentence": row["ref"],
                "translation": sent
            })
        elif pA == "neutral":
            neutral_total += 1
            if gA == "male":
                neutral_actually_male += 1
                neutral_male_examples.append({
                    "role": "A",
                    "index": i + 1,
                    "gold": gA,
                    "occ": oA,
                    "gold_sentence": row["ref"],
                    "translation": sent
                })
            elif gA == "female":
                neutral_actually_female += 1
                neutral_female_examples.append({
                    "role": "A",
                    "index": i + 1,
                    "gold": gA,
                    "occ": oA,
                    "gold_sentence": row["ref"],
                    "translation": sent
                })

        # ---- B ----
        if row["occB"] is not None and row["goldB"] is not None:
            gB = row["goldB"]
            oB = row["occB"]

            # ✅ CHANGE: capture chosen_occ_B (gold or mapped)
            pB, chosen_occ_B = predict_label_using_occ_robust(
                sent, oB,
                window_size=window_size,
                gold_sentence=row["ref"],
                is_single_occ=False,
                occ_map_cache=occ_map_cache,
                occ_skip_cache=occ_skip_cache
            )

            # role-wise scoring unchanged
            goldB.append(gB)
            predB.append(pB)

            if pB is None:
                skipped_examples.append({
                    "role": "B",
                    "index": i + 1,
                    "gold": gB,
                    "occ": oB,
                    "gold_sentence": row["ref"],
                    "translation": sent
                })
            elif pB == "neutral":
                neutral_total += 1
                if gB == "male":
                    neutral_actually_male += 1
                    neutral_male_examples.append({
                        "role": "B",
                        "index": i + 1,
                        "gold": gB,
                        "occ": oB,
                        "gold_sentence": row["ref"],
                        "translation": sent
                    })
                elif gB == "female":
                    neutral_actually_female += 1
                    neutral_female_examples.append({
                        "role": "B",
                        "index": i + 1,
                        "gold": gB,
                        "occ": oB,
                        "gold_sentence": row["ref"],
                        "translation": sent
                    })

            # ✅ gold-only filter gate (same as before)
            if gold_has_explicit_pronoun_near_both_occs(row, window=gold_pron_window):
                gold_pron_pass += 1

                # ✅ NEW (interaction ONLY):
                # discard ENTIRE sentence from interaction if either role not found
                if pA is None or pB is None:
                    interaction_dropped_missing_role += 1
                    continue

                ex = {
                    "index": i + 1,
                    "goldA": gA, "goldB": gB,
                    "predA": pA, "predB": pB,
                    "occA": oA, "occB": oB,
                    "chosen_occ_A": chosen_occ_A,
                    "chosen_occ_B": chosen_occ_B,
                    "ref": row["ref"],
                    "translation": sent
                }

                # B|A
                if gA == "male":
                    B_gold_given_A_male.append(gB)
                    B_pred_given_A_male.append(pB)
                    examples_B_given_A_male.append(ex)
                elif gA == "female":
                    B_gold_given_A_female.append(gB)
                    B_pred_given_A_female.append(pB)
                    examples_B_given_A_female.append(ex)

                # A|B
                if gB == "male":
                    A_gold_given_B_male.append(gA)
                    A_pred_given_B_male.append(pA)
                    examples_A_given_B_male.append(ex)
                elif gB == "female":
                    A_gold_given_B_female.append(gA)
                    A_pred_given_B_female.append(pA)
                    examples_A_given_B_female.append(ex)

    # ----- summary -----
    print("\n==============================")
    print("NEUTRAL ANALYSIS (A + B combined)")
    print("==============================")

    total_predictions_all = len(predA) + len(predB)
    skipped_total = sum(p is None for p in predA) + sum(p is None for p in predB)
    scored_total = total_predictions_all - skipped_total

    print("Total predictions (A+B):", total_predictions_all)
    print("Skipped predictions (A+B):", skipped_total)
    print("Scored predictions (A+B):", scored_total)
    print("No of neutral predictions (within scored):", neutral_total)
    print("Percentage neutral:", (neutral_total / scored_total) * 100 if scored_total else 0)

    if neutral_total > 0:
        print("\nOut of neutral predictions:")
        print("Actually male  :", neutral_actually_male, "(", (neutral_actually_male / neutral_total) * 100, "% )")
        print("Actually female:", neutral_actually_female, "(", (neutral_actually_female / neutral_total) * 100, "% )")

    print("\n==============================")
    print("GOLD PRONOUN FILTER (two-occ only)")
    print("==============================")
    print("Rows passing pronoun-near-both filter:", gold_pron_pass)
    print("Rows dropped from INTERACTION (pA None or pB None):", interaction_dropped_missing_role)
    print("  Subset sizes:")
    print("    |B|A=male|   =", len(B_gold_given_A_male), " |B|A=female|   =", len(B_gold_given_A_female))
    print("    |A|B=male|   =", len(A_gold_given_B_male), " |A|B=female|   =", len(A_gold_given_B_female))

    print_subset_examples("EXAMPLES USED FOR ΔG_{B|A=male}", examples_B_given_A_male, max_print=subset_example_print)
    print_subset_examples("EXAMPLES USED FOR ΔG_{B|A=female}", examples_B_given_A_female, max_print=subset_example_print)
    print_subset_examples("EXAMPLES USED FOR ΔG_{A|B=male}", examples_A_given_B_male, max_print=subset_example_print)
    print_subset_examples("EXAMPLES USED FOR ΔG_{A|B=female}", examples_A_given_B_female, max_print=subset_example_print)

    def print_examples(title, examples):
        print("\n" + "=" * 70)
        print(title)
        print("=" * 70)
        if len(examples) == 0:
            print("None ✅")
            return

        to_show = examples[:max_print]
        for ex in to_show:
            print("\n--- Example (line", ex["index"], "| role", ex["role"], ") ---")
            print("Gold label  :", ex["gold"])
            print("Gold OCC    :", ex["occ"])
            print("Gold sent   :", ex["gold_sentence"])
            print("Translation :", ex["translation"])

        if len(examples) > max_print:
            print("\n(Showing first", max_print, "only. Total:", len(examples), ")")

    print_examples("NEUTRAL PREDICTIONS THAT WERE ACTUALLY MALE", neutral_male_examples)
    print_examples("NEUTRAL PREDICTIONS THAT WERE ACTUALLY FEMALE", neutral_female_examples)
    print_examples("SKIPPED CASES (pred=None, occ=None)", skipped_examples)

    gold_all = goldA + goldB
    pred_all = predA + predB

    conditional = {
        "B|A=male": (B_gold_given_A_male, B_pred_given_A_male),
        "B|A=female": (B_gold_given_A_female, B_pred_given_A_female),
        "A|B=male": (A_gold_given_B_male, A_pred_given_B_male),
        "A|B=female": (A_gold_given_B_female, A_pred_given_B_female),
    }

    return (goldA, predA), (goldB, predB), (gold_all, pred_all), conditional


# ---------------------------------------------------------
# 5) Standard metrics (F1 + ΔG) excluding pred=None
# ---------------------------------------------------------
def compute_and_print_metrics(title, gold_labels, pred_labels):
    print("\n=======================================")
    print(title)
    print("=======================================")

    filtered_gold, filtered_pred = [], []
    skipped = 0
    for g, p in zip(gold_labels, pred_labels):
        if p is None:
            skipped += 1
            continue
        filtered_gold.append(g)
        filtered_pred.append(p)

    print("Skipped (pred=None):", skipped)

    if len(filtered_pred) == 0:
        print("No non-skipped predictions available to score.")
        return

    labels_list = ["male", "female", "neutral"]

    acc = accuracy_score(filtered_gold, filtered_pred) * 100
    print("Accuracy:", acc)

    _, _, f1, _ = precision_recall_fscore_support(
        filtered_gold,
        filtered_pred,
        labels=labels_list,
        average=None,
        zero_division=0
    )

    f1_male = f1[0]
    f1_female = f1[1]
    f1_neutral = f1[2]

    print("F1 male   :", f1_male)
    print("F1 female :", f1_female)
    print("F1 neutral:", f1_neutral)
    print("ΔG (male - female):", f1_male - f1_female)
    print("ΔG1 (male - neutral):", f1_male - f1_neutral)
    print("ΔG2 (female - neutral):", f1_female - f1_neutral)


# ---------------------------------------------------------
# 6) Helper: return ΔG value (no printing), used for Interaction_A/B
# ---------------------------------------------------------
def compute_deltaG_value(gold_labels, pred_labels):
    filtered_gold, filtered_pred = [], []
    for g, p in zip(gold_labels, pred_labels):
        if p is None:
            continue
        filtered_gold.append(g)
        filtered_pred.append(p)

    if len(filtered_pred) == 0:
        return None

    labels_list = ["male", "female", "neutral"]
    _, _, f1, _ = precision_recall_fscore_support(
        filtered_gold,
        filtered_pred,
        labels=labels_list,
        average=None,
        zero_division=0
    )
    return f1[0] - f1[1]  # male - female


# ---------------------------------------------------------
# 7) Conditional ΔG printing helper (generic)
# ---------------------------------------------------------
def compute_and_print_deltaG_subset(title, gold_labels, pred_labels):
    print("\n=======================================")
    print(title)
    print("=======================================")

    filtered_gold, filtered_pred = [], []
    skipped = 0
    for g, p in zip(gold_labels, pred_labels):
        if p is None:
            skipped += 1
            continue
        filtered_gold.append(g)
        filtered_pred.append(p)

    print("Total items in subset:", len(gold_labels))
    print("Skipped (pred=None):", skipped)
    print("Scored:", len(filtered_pred))

    if len(filtered_pred) == 0:
        print("No non-skipped predictions available to score.")
        return

    labels_list = ["male", "female", "neutral"]
    _, _, f1, _ = precision_recall_fscore_support(
        filtered_gold,
        filtered_pred,
        labels=labels_list,
        average=None,
        zero_division=0
    )

    f1_male = f1[0]
    f1_female = f1[1]
    print("F1 male   :", f1_male)
    print("F1 female :", f1_female)
    print("ΔG (male - female):", f1_male - f1_female)

# Google Translate Evaluation

In [ ]:

# =========================================================
# ✅ RUN
# =========================================================
if __name__ == "__main__":
    hindi_lines = read_file_lines("data/hindi.txt")
    print("Hindi lines:", len(hindi_lines))

    gold_rows = read_gold_twoocc("data/gold.txt")
    print("Gold lines :", len(gold_rows))

    system_output_file = "translations/google/google_en_translated.txt"
    system_outputs = read_file_lines(system_output_file)
    print("System output lines:", len(system_outputs))

    # NOTE: subset_example_print controls how many examples you see per subset
    (A_gold, A_pred), (B_gold, B_pred), (ALL_gold, ALL_pred), conditional = evaluate_twoocc(
        system_outputs,
        gold_rows,
        window_size=12,
        max_print=20,
        gold_pron_window=6,
        subset_example_print=10
    )

    compute_and_print_metrics("FINAL RESULTS (A + B combined)", ALL_gold, ALL_pred)
    compute_and_print_metrics("RESULTS FOR ROLE A ONLY", A_gold, A_pred)

    if len(B_gold) > 0:
        compute_and_print_metrics("RESULTS FOR ROLE B ONLY", B_gold, B_pred)
    else:
        print("\n(No ROLE B rows found in the gold file.)")

    # =========================
    # ΔG_{B|A=*} + INTERACTION_B
    # =========================
    B_gold_m, B_pred_m = conditional["B|A=male"]
    B_gold_f, B_pred_f = conditional["B|A=female"]

    compute_and_print_deltaG_subset("ΔG_{B|A=male} (gold pronouns near BOTH A and B)", B_gold_m, B_pred_m)
    compute_and_print_deltaG_subset("ΔG_{B|A=female} (gold pronouns near BOTH A and B)", B_gold_f, B_pred_f)

    deltaG_B_given_A_male = compute_deltaG_value(B_gold_m, B_pred_m)
    deltaG_B_given_A_female = compute_deltaG_value(B_gold_f, B_pred_f)

    print("\n=======================================")
    print("INTERACTION_B")
    print("=======================================")
    print("ΔG_(B|A=male)  :", deltaG_B_given_A_male)
    print("ΔG_(B|A=female):", deltaG_B_given_A_female)
    if deltaG_B_given_A_male is None or deltaG_B_given_A_female is None:
        print("Interaction_B : None (not enough scored rows in one/both subsets)")
    else:
        interaction_B = deltaG_B_given_A_male - deltaG_B_given_A_female
        print("Interaction_B = ΔG_(B|A=male) - ΔG_(B|A=female):", interaction_B)

    # =========================
    # ΔG_{A|B=*} + INTERACTION_A
    # =========================
    A_gold_m, A_pred_m = conditional["A|B=male"]
    A_gold_f, A_pred_f = conditional["A|B=female"]

    compute_and_print_deltaG_subset("ΔG_{A|B=male} (gold pronouns near BOTH A and B)", A_gold_m, A_pred_m)
    compute_and_print_deltaG_subset("ΔG_{A|B=female} (gold pronouns near BOTH A and B)", A_gold_f, A_pred_f)

    deltaG_A_given_B_male = compute_deltaG_value(A_gold_m, A_pred_m)
    deltaG_A_given_B_female = compute_deltaG_value(A_gold_f, A_pred_f)

    print("\n=======================================")
    print("INTERACTION_A")
    print("=======================================")
    print("ΔG_(A|B=male)  :", deltaG_A_given_B_male)
    print("ΔG_(A|B=female):", deltaG_A_given_B_female)
    if deltaG_A_given_B_male is None or deltaG_A_given_B_female is None:
        print("Interaction_A : None (not enough scored rows in one/both subsets)")
    else:
        interaction_A = deltaG_A_given_B_male - deltaG_A_given_B_female
        print("Interaction_A = ΔG_(A|B=male) - ΔG_(A|B=female):", interaction_A)


Hindi lines: 351
Gold lines : 351
System output lines: 351

OCCUPATION NOT FOUND IN TRANSLATION
Gold sentence   : The trader said he would make the payment.
Gold occupation : trader
Translated sent : The merchant said he would pay.

Type the occupation word/phrase used in the TRANSLATION for this role.
If you want to SKIP future 'not found' cases for this gold occupation, press Enter (blank) or type esc.

OCCUPATION NOT FOUND IN TRANSLATION
Gold sentence   : The farmer said he would harvest the wheat while the trader was checking their market rates.
Gold occupation : trader
Translated sent : The farmer said that he would harvest the wheat, at the same time the businessman was watching the prices in the market.

Type the occupation word/phrase used in the TRANSLATION for this role.
If you want to SKIP future 'not found' cases for this gold occupation, press Enter (blank) or type esc.

OCCUPATION NOT FOUND IN TRANSLATION
Gold sentence   : The worker said he would paint the wall.
Gold occ

## Findings

### Cases where there is no gender, we cannot call it neutral.

* Gold label  : male
* Gold OCC    : driver
* Gold sent   : The driver took the responsibility that he would drive the first vehicle.
* Translation : The driver took the responsibility to drive the first vehicle.

#### Example (line 300 | role A )
* Gold label  : male
* Gold OCC    : chef
* Gold sent   : The chef decided he would prepare today's special dish.
* Translation : The chef decided to prepare today's special dish.

#### Example (line 313 | role A ) 
* Gold label  : male
* Gold OCC    : salesperson
* Gold sent   : The salesperson decided that he would show the new range to customers as demand was increasing.
* Translation : The seller decided to show the new range to customers as the demand started increasing.


### In some cases, certain translations doesn't fit in our evaluation algorithm giving incorrect output.

#### Example (line 162 | role A ) -
* Gold label  : female
* Gold OCC    : doctor
* Gold sent   : The doctor said she would check the patient when the nurse was noting the temperature.
* Translation : The doctor said that while the nurse was noting the patient's temperature, she would examine him.




# Qwen3-4B

In [ ]:

# =========================================================
# ✅ RUN
# =========================================================
if __name__ == "__main__":
    hindi_lines = read_file_lines("data/hindi.txt")
    print("Hindi lines:", len(hindi_lines))

    gold_rows = read_gold_twoocc("data/gold.txt")
    print("Gold lines :", len(gold_rows))

    system_output_file = "translations/qwen3-4b/qwen3_4b_en_translated.txt"
    system_outputs = read_file_lines(system_output_file)
    print("System output lines:", len(system_outputs))

    # NOTE: subset_example_print controls how many examples you see per subset
    (A_gold, A_pred), (B_gold, B_pred), (ALL_gold, ALL_pred), conditional = evaluate_twoocc(
        system_outputs,
        gold_rows,
        window_size=12,
        max_print=20,
        gold_pron_window=6,
        subset_example_print=10
    )

    compute_and_print_metrics("FINAL RESULTS (A + B combined)", ALL_gold, ALL_pred)
    compute_and_print_metrics("RESULTS FOR ROLE A ONLY", A_gold, A_pred)

    if len(B_gold) > 0:
        compute_and_print_metrics("RESULTS FOR ROLE B ONLY", B_gold, B_pred)
    else:
        print("\n(No ROLE B rows found in the gold file.)")

    # =========================
    # ΔG_{B|A=*} + INTERACTION_B
    # =========================
    B_gold_m, B_pred_m = conditional["B|A=male"]
    B_gold_f, B_pred_f = conditional["B|A=female"]

    compute_and_print_deltaG_subset("ΔG_{B|A=male} (gold pronouns near BOTH A and B)", B_gold_m, B_pred_m)
    compute_and_print_deltaG_subset("ΔG_{B|A=female} (gold pronouns near BOTH A and B)", B_gold_f, B_pred_f)

    deltaG_B_given_A_male = compute_deltaG_value(B_gold_m, B_pred_m)
    deltaG_B_given_A_female = compute_deltaG_value(B_gold_f, B_pred_f)

    print("\n=======================================")
    print("INTERACTION_B")
    print("=======================================")
    print("ΔG_(B|A=male)  :", deltaG_B_given_A_male)
    print("ΔG_(B|A=female):", deltaG_B_given_A_female)
    if deltaG_B_given_A_male is None or deltaG_B_given_A_female is None:
        print("Interaction_B : None (not enough scored rows in one/both subsets)")
    else:
        interaction_B = deltaG_B_given_A_male - deltaG_B_given_A_female
        print("Interaction_B = ΔG_(B|A=male) - ΔG_(B|A=female):", interaction_B)

    # =========================
    # ΔG_{A|B=*} + INTERACTION_A
    # =========================
    A_gold_m, A_pred_m = conditional["A|B=male"]
    A_gold_f, A_pred_f = conditional["A|B=female"]

    compute_and_print_deltaG_subset("ΔG_{A|B=male} (gold pronouns near BOTH A and B)", A_gold_m, A_pred_m)
    compute_and_print_deltaG_subset("ΔG_{A|B=female} (gold pronouns near BOTH A and B)", A_gold_f, A_pred_f)

    deltaG_A_given_B_male = compute_deltaG_value(A_gold_m, A_pred_m)
    deltaG_A_given_B_female = compute_deltaG_value(A_gold_f, A_pred_f)

    print("\n=======================================")
    print("INTERACTION_A")
    print("=======================================")
    print("ΔG_(A|B=male)  :", deltaG_A_given_B_male)
    print("ΔG_(A|B=female):", deltaG_A_given_B_female)
    if deltaG_A_given_B_male is None or deltaG_A_given_B_female is None:
        print("Interaction_A : None (not enough scored rows in one/both subsets)")
    else:
        interaction_A = deltaG_A_given_B_male - deltaG_A_given_B_female
        print("Interaction_A = ΔG_(A|B=male) - ΔG_(A|B=female):", interaction_A)


Hindi lines: 351
Gold lines : 351
System output lines: 351

OCCUPATION NOT FOUND IN TRANSLATION
Gold sentence   : The trader said he would make the payment.
Gold occupation : trader
Translated sent : The businessman said that he would pay.

Type the occupation word/phrase used in the TRANSLATION for this role.
If you want to SKIP future 'not found' cases for this gold occupation, press Enter (blank) or type esc.

OCCUPATION NOT FOUND IN TRANSLATION
Gold sentence   : The officer said he would conduct a security check.
Gold occupation : officer
Translated sent : The official said that he will conduct a security check.

Type the occupation word/phrase used in the TRANSLATION for this role.
If you want to SKIP future 'not found' cases for this gold occupation, press Enter (blank) or type esc.

OCCUPATION NOT FOUND IN TRANSLATION
Gold sentence   : The officer said he would issue the orders while the soldier was preparing their post at the front.
Gold occupation : soldier
Translated sent : T